# EfficientNetV2S — TensorFlow / Keras

Fused-MBConv (stages 0-2) + MBConv with SE (stages 3-5). Input: **300×300**. ~21.6M params.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import roc_auc_score
print('TF:', tf.__version__)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def _conv_bn(x, filters, kernel_size, strides=1, padding="same", activation="swish"):
    x = layers.Conv2D(filters, kernel_size, strides=strides, padding=padding, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if activation:
        x = layers.Activation(activation)(x)
    return x

def _se_block(x, in_ch, se_ratio):
    """Squeeze-Excitation: bottleneck = in_ch * se_ratio."""
    se_ch = max(1, int(in_ch * se_ratio))
    ch    = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Reshape((1, 1, ch))(se)
    se = layers.Conv2D(se_ch, 1, activation="swish",   use_bias=True)(se)
    se = layers.Conv2D(ch,    1, activation="sigmoid", use_bias=True)(se)
    return layers.Multiply()([x, se])

def _fused_mbconv(x, in_ch, out_ch, expand, stride, se_ratio=0.0):
    """Fused-MBConv: single 3x3 conv replaces expand+DWConv (used in early stages)."""
    shortcut = x
    if expand != 1:
        x = _conv_bn(x, in_ch * expand, 3, strides=stride)  # fused expand (with act)
        x = _conv_bn(x, out_ch, 1, activation=None)          # project (no act)
    else:
        x = _conv_bn(x, out_ch, 3, strides=stride)           # single conv (with act)
    if se_ratio > 0:
        x = _se_block(x, in_ch, se_ratio)
    if stride == 1 and in_ch == out_ch:
        x = layers.Add()([shortcut, x])
    return x

def _mbconv(x, in_ch, out_ch, expand, stride, se_ratio=0.25):
    """MBConv with Squeeze-Excitation (used in later stages)."""
    shortcut = x
    x = _conv_bn(x, in_ch * expand, 1)                                                 # expand
    x = layers.DepthwiseConv2D(3, strides=stride, padding="same", use_bias=False)(x)   # depthwise
    x = layers.BatchNormalization()(x)
    x = layers.Activation("swish")(x)
    if se_ratio > 0:
        x = _se_block(x, in_ch, se_ratio)
    x = _conv_bn(x, out_ch, 1, activation=None)                                        # project
    if stride == 1 and in_ch == out_ch:
        x = layers.Add()([shortcut, x])
    return x

def _build_stages(x, block_args):
    for stage in block_args:
        in_ch, out_ch = stage["in_ch"], stage["out_ch"]
        for i in range(stage["n"]):
            s  = stage["stride"] if i == 0 else 1
            ic = in_ch           if i == 0 else out_ch
            if stage["fused"]:
                x = _fused_mbconv(x, ic, out_ch, stage["expand"], s, stage["se"])
            else:
                x = _mbconv(x, ic, out_ch, stage["expand"], s, stage["se"])
    return x

# EfficientNetV2-S block configuration
BLOCK_ARGS_S = [
    {"n": 2,  "in_ch": 24,  "out_ch": 24,  "expand": 1, "stride": 1, "se": 0.0,  "fused": True},
    {"n": 4,  "in_ch": 24,  "out_ch": 48,  "expand": 4, "stride": 2, "se": 0.0,  "fused": True},
    {"n": 4,  "in_ch": 48,  "out_ch": 64,  "expand": 4, "stride": 2, "se": 0.0,  "fused": True},
    {"n": 6,  "in_ch": 64,  "out_ch": 128, "expand": 4, "stride": 2, "se": 0.25, "fused": False},
    {"n": 9,  "in_ch": 128, "out_ch": 160, "expand": 6, "stride": 1, "se": 0.25, "fused": False},
    {"n": 15, "in_ch": 160, "out_ch": 256, "expand": 6, "stride": 2, "se": 0.25, "fused": False},
]

def build_efficientnetv2s(num_classes=1000, input_shape=(300, 300, 3)):
    inputs = keras.Input(shape=input_shape)
    x = _conv_bn(inputs, 24, 3, strides=2)          # stem: 300->150
    x = _build_stages(x, BLOCK_ARGS_S)
    x = _conv_bn(x, 1280, 1)                         # head conv
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, x, name="EfficientNetV2S")

In [ ]:
model = build_efficientnetv2s(num_classes=10)
model.summary()

In [ ]:
dummy = tf.random.normal([2, 300, 300, 3])
out = model(dummy, training=False)
print('Output shape:', out.shape)

In [ ]:
total = sum(p.numpy().size for p in model.trainable_weights)
print(f'Trainable params: {total:,}')

In [ ]:
layer_info = [(l.name, l.__class__.__name__, l.count_params()) for l in model.layers]
for name, typ, params in layer_info[-20:]:
    print(f'{name:45s}  {typ:30s}  {params:>10,}')

## Training

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

BATCH = 16
IMG_SIZE = (300, 300)
TRAIN_DIR = "dataset/train"
VAL_DIR   = "dataset/val"

train_gen = ImageDataGenerator(
    rescale=1./255, rotation_range=20, width_shift_range=0.1,
    height_shift_range=0.1, horizontal_flip=True, zoom_range=0.1,
)
val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH, class_mode="categorical")
val_data   = val_gen.flow_from_directory(VAL_DIR,   target_size=IMG_SIZE, batch_size=BATCH, class_mode="categorical", shuffle=False)
NUM_CLASSES = len(train_data.class_indices)
print("Classes:", NUM_CLASSES)


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

model = build_efficientnetv2s(num_classes=NUM_CLASSES)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
callbacks = [
    ModelCheckpoint("efficientnetv2s_best.keras", monitor="val_accuracy", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.1, patience=5, min_lr=1e-7, verbose=1),
]
history = model.fit(train_data, validation_data=val_data, epochs=30, callbacks=callbacks)


## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train"); axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy"); axes[0].legend()
axes[1].plot(history.history["loss"], label="train"); axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss"); axes[1].legend()
plt.tight_layout(); plt.show()


## Inference

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

CLASS_NAMES = list(train_data.class_indices.keys())

def predict_image(img_path, model, class_names, img_size=(300, 300)):
    img = load_img(img_path, target_size=img_size)
    x = img_to_array(img) / 255.0
    x = np.expand_dims(x, 0)
    probs = model.predict(x, verbose=0)[0]
    idx = np.argmax(probs)
    print(f"Prediction: {class_names[idx]}  ({probs[idx]*100:.1f}%)")
    return class_names[idx], probs

# predict_image("test.jpg", model, CLASS_NAMES)


## ROC-AUC

In [ ]:
val_gen.reset()
y_true = val_data.classes
y_prob = model.predict(val_data, verbose=1)

In [ ]:
from sklearn.preprocessing import label_binarize

y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
auc   = roc_auc_score(y_bin, y_prob, average="macro", multi_class="ovr")
print(f"Macro ROC-AUC: {auc:.4f}")

from sklearn.metrics import roc_curve, auc as auc_score
plt.figure(figsize=(8, 6))
for i in range(min(NUM_CLASSES, 10)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, lw=1, label=f"{CLASS_NAMES[i]} (AUC={auc_score(fpr,tpr):.2f})")
plt.plot([0,1],[0,1],"k--"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC Curves"); plt.legend(fontsize=7); plt.show()
